# Before you model: what a data set can support

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igbaskin/cmf-qsar/blob/main/docs/tutorials/12_before_you_model.ipynb)

The notebooks from the sixth to the eleventh each made the same checks by hand. Related samples
were held out together. The model was compared with counts of atoms or with one number. The
question was asked whether a frame adds anything. Every reference was given a search of its own
settings.

This notebook is about the tool that makes these checks for a table of molecules and activities,
before any model is built: the function `diagnose`, which is `cmf diagnose` on the command line.
It is not a model and it does not improve one. It answers five questions.

1. **Is there anything to model?** The number of molecules and the spread of the activity.
2. **Can a frame exist?** How many different scaffolds the set has, and whether the molecules
   are superimposed.
3. **Does the frame carry signal?** The model in the frame of the molecules against two routes
   that need no frame: counts of atom types, and the rotation-invariant kernel of the fields.
4. **Does the geometry add anything?** The same model against a few descriptors that need no
   three-dimensional structure.
5. **Was the model specified sensibly?** Whether the chosen attenuation lies at the edge of the
   values that were tried.

Then it recommends a route. The notebook runs it on four sets from the repository whose answers
differ, reads one report line by line, and ends with what the recommendation is worth.

Running time: about two minutes.

In [1]:
# Run this cell first. On Google Colab it fetches the repository and installs the package;
# on a local checkout it only moves to the repository root, so that the example paths work.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("cmf-qsar"):
        subprocess.run(["git", "clone", "-q", "https://github.com/igbaskin/cmf-qsar.git"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./cmf-qsar", "py3Dmol"], check=True)
    os.chdir("cmf-qsar")
else:
    while not os.path.exists("pyproject.toml") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")
print("working in", os.getcwd())

working in C:\Users\baski\work\claude\cmf


## One report, read line by line

The first set is the thrombin series of the Sutherland collection: 88 inhibitors, superimposed
by the authors of the set, with a measured pK<sub>i</sub>. The charges are read from the file.

In [2]:
import json
import numpy as np
from rdkit import RDLogger
from cmfqsar import load_dataset
from cmfqsar.diagnose import diagnose

RDLogger.DisableLog("rdApp.*")
ROUTES = ("aligned", "counts", "invariant", "descriptors")       # the routes of `cmf diagnose`


def ask(path, charges):
    """The diagnosis of one file, as the command line runs it."""
    data = load_dataset(path)
    y = np.asarray(data.activity, float)
    return diagnose(data.mols, data.names, y, charge_model=charges, routes=ROUTES,
                    activity_prop=data.activity_prop), data, y


reports = {}
reports["thrombin"], thrombin, y_thrombin = ask("examples/sutherland/THR.sdf", "file")
print(reports["thrombin"].report())

88 molecules, activity from 4.36 to 8.48 (sd 1.01)

Hyper-parameters chosen inside each fold and scored outside it.

How well each route predicts, 10-fold CV:
  aligned      q2 +0.616  RMSEcv 0.624  (alpha 0.4, lambda 0.1)
  counts       q2 +0.051  RMSEcv 0.980  (lambda 0.1)
  invariant    q2 +0.511  RMSEcv 0.703  (lambda 0.1)   width of its distance axis 0.75 A, not searched
  descriptors  q2 +0.057  RMSEcv 0.977  (lambda 1)   4 descriptors

What the set says:
! 40 chemotype(s); the largest holds 16 of 88 molecules -- a series of several scaffolds has no common frame, and an alignment will invent one
- the common framework (framework of the largest chemotype) has 30 heavy atoms, about 83% of a typical molecule, and 25% of the set can be cut at it
  the 30 shared framework atoms sit 0.0 A apart between molecules, over the best of their matches (22 of 88 carry them)
  the aligned model is +0.564 in q2 against counts (95% interval +0.366 to +0.810) -- the frame does better than counts
! 

The report has three parts.

**How well each route predicts.** Four ways of comparing the molecules, each with a kernel ridge
regression and a ten-fold cross-validation. The settings of every route are chosen inside each
training part, so the numbers are comparable although the routes have different numbers of
settings.

- `aligned` is the ordinary model of the first notebook: five fields in the frame in which the
  molecules arrive.
- `counts` knows how many atoms of each type a molecule has, and nothing about its geometry.
- `invariant` is the rotation-invariant kernel of the second notebook: the same fields, compared
  through the distances between atoms, so no frame is needed.
- `descriptors` is a regression on a few numbers that need no three-dimensional structure, such
  as the molecular weight and counts of hydrogen-bond donors and acceptors.

**What the set says.** One line for every finding. A line that starts with `!` is a warning, a
line with `-` a note. For thrombin there are two warnings, and they show how to read them.

- *Forty chemotypes.* The program could not have aligned this set by itself, because there is no
  common scaffold. The next line says that the molecules are superimposed all the same: the
  shared atoms lie 0.0 Å apart. So a frame exists. The authors made it, by docking.
- *The frame gains nothing over the invariant route.* The aligned model is ahead by 0.10, and the
  interval of that difference reaches just below zero. With 88 molecules a difference of 0.10 is
  at the limit of what can be seen.

The other two comparisons are clear: the frame is far ahead of the counts and of the
descriptors, with intervals well above zero. So the selectivity of thrombin for these inhibitors
depends on where the groups are, and not only on which groups there are.

**The recommendation** follows from these lines: the aligned route, because the frame carries
signal and the molecules are superimposed. It is a prompt and not a verdict. It says which route
to take, and it never says that a model is good.

## Four sets, four answers

The same function on three more sets. Two are from the same collection, inhibitors of the
angiotensin-converting enzyme (ACE) and ligands of the benzodiazepine receptor (BZR). The fourth
is a table of corrosion inhibitors with one heterocycle in common, given as SMILES, so the
program has to make the three-dimensional structures itself.

In [3]:
reports["ACE"], ace, y_ace = ask("examples/sutherland/ACE.sdf", "file")
reports["benzodiazepine receptor"], bzr, y_bzr = ask("examples/sutherland/BZR.sdf", "file")
reports["corrosion, thiadiazoles"], corrosion, y_corrosion = ask("examples/corrosion/7_c1nncs1.csv", "gasteiger")

print("%-26s %4s %9s %8s %10s %12s %14s   %s" % ("set", "n", "aligned", "counts", "invariant", "descriptors", "file columns", "recommended"))
for name, d in reports.items():
    q2 = [d.routes[r].q2 for r in ROUTES]
    own = "%+.2f" % d.routes["file columns"].q2 if "file columns" in d.routes else "none"
    print("%-26s %4d %+9.2f %+8.2f %+10.2f %+12.2f %14s   %s" % ((name, d.n) + tuple(q2) + (own, d.recommended)))

set                           n   aligned   counts  invariant  descriptors   file columns   recommended
thrombin                     88     +0.62    +0.05      +0.51        +0.06           none   aligned, with the frame checked
ACE                         114     +0.60    +0.66      +0.68        +0.64           none   descriptors
benzodiazepine receptor     147     +0.28    +0.20      +0.41        +0.13           none   invariant
corrosion, thiadiazoles      29     -0.09    -0.23      -0.19        -0.13          +0.99   none


Four sets and four different recommendations. Each is read below. The column `file columns`
is not a route to choose: it is explained with the fourth set.

### ACE: descriptors do as well

On this set all four routes reach about the same score, and the aligned model is the lowest of
them. The warnings say that the frame gains nothing over any other route.

In [4]:
for finding in reports["ACE"].findings:
    if finding.level == "warn" and "aligned model" in finding.text:
        print(finding)
print()
print("recommended:", reports["ACE"].recommended)
print("  ", reports["ACE"].why)
descriptors = reports["ACE"].routes["descriptors"]
print()
print("the descriptor route:", descriptors.note)
print("its descriptors with the highest correlation with the activity:",
      ", ".join("%s (r %+.2f)" % pair for pair in descriptors.correlations[:4]))

! the aligned model is -0.056 in q2 against counts (95% interval -0.179 to +0.052) -- the frame gains nothing over counts
! the aligned model is -0.072 in q2 against invariant (95% interval -0.202 to +0.020) -- the frame gains nothing over invariant
! the aligned model is -0.040 in q2 against descriptors (95% interval -0.170 to +0.065) -- descriptors alone do as well, so the geometry adds nothing on this set

recommended: descriptors
   descriptors alone match the 3D model, so a field model here is an expensive way to count atoms; use the fields only if the co-field is the purpose

the descriptor route: 4 descriptors
its descriptors with the highest correlation with the activity: calc:mw (r +0.76), calc:rotatable (r +0.61), calc:tpsa (r +0.54), calc:clogp (r +0.30)


The inhibitors of this set run from small dipeptide mimics to large molecules, and the activity
grows with the size. A count of atoms knows the size, so the three-dimensional model has nothing
to add in prediction. This does not make the field model useless here. Its co-field can still
be drawn and read, and the recommendation says so. But its score must not be presented as
evidence that the geometry matters.

### The benzodiazepine receptor: no frame needed

Here the route without a frame is the best one, and the aligned model is not ahead of it. The
recommendation is the invariant route, and its reason is practical: an alignment that gains
nothing is work that can be done wrongly.

One thing about this route has to be known. The invariant kernel has a setting of its own, the
width of its distance axis, and the diagnosis does not search it: it runs the default of 0.75 Å
and says so in the report. The catalyst series of the ninth notebook showed that such a setting
can decide a comparison. So it is tried here by hand.

In [5]:
from cmfqsar.diagnose import invariant_route

print("the aligned route on this set:      q2 %+.2f" % reports["benzodiazepine receptor"].routes["aligned"].q2)
for width in (0.25, 0.5, 0.75, 1.5):
    route = invariant_route(bzr.mols, bzr.names, y_bzr, charge_model="file", width=width)
    print("the invariant route, width %-5g A: q2 %+.2f" % (width, route.q2))

the aligned route on this set:      q2 +0.28


the invariant route, width 0.25  A: q2 +0.46


the invariant route, width 0.5   A: q2 +0.38


the invariant route, width 0.75  A: q2 +0.41


the invariant route, width 1.5   A: q2 +0.33


On this set the width changes the score a little, and no value changes the reading: the
invariant route stays at or above the aligned one. On molecules of this size the default is a
fair choice. On a series of much larger molecules it was not, and there a smaller width was far
better. When the recommendation depends on the comparison with this route, a few lines like
these are worth running.

### The thiadiazoles: nothing to model, and the columns of the file

The fourth set is the instructive one. It has 29 molecules, and no route that describes the
molecules predicts the activity. Still one number in its row of the table is 0.99.

In [6]:
d = reports["corrosion, thiadiazoles"]
for route in d.routes.values():
    print("%-13s q2 %+.2f   %s" % (route.name, route.q2, route.note))
print()
for finding in d.findings:
    if finding.code in ("superimposed", "file-columns"):
        print(finding)
        print()
print("recommended:", d.recommended)
print("  ", d.why)

aligned       q2 -0.09   
counts        q2 -0.23   
invariant     q2 -0.19   width of its distance axis 0.75 A, not searched
descriptors   q2 -0.13   4 descriptors
file columns  q2 +0.99   10 descriptors; dropped as the activity under another name: mwila_pooled_raw; not used for the recommendation

! the 20 shared framework atoms sit 1.3 A apart between molecules, over the best of their matches (3 of 29 carry them) -- the molecules are not in a common frame

! the file has 11 numeric column(s) of its own (mw, n_papers, dG_ads, depth, mwila_hcl_adj, mwila_h2so4_raw, mwila_h2so4_adj, mwila_pooled_raw and 3 more); with them the descriptor baseline is +1.123 in q2 against the computed descriptors (95% interval +0.955 to +1.547); the activity under another name, by a correlation above 0.95: mwila_pooled_raw -- these columns predict the activity, so they are measurements or forms of the activity and not descriptions of the structure. They are not used. If some of them are real descriptors, s

Two warnings carry the lesson.

- **The molecules are not in a common frame.** The file holds SMILES, the structures were
  generated one by one, and nothing superimposed them. An aligned model of such a set compares
  molecules in arbitrary orientations.
- **The columns of the file predict the activity.** The table has eleven numeric columns besides
  the activity. With them a descriptor baseline reaches a q² of 0.99, where the descriptors
  computed from the structure reach nothing. These columns are not descriptions of the
  molecules. One is the activity under another name, and others are adjusted forms of the same
  measurement, which rebuild it between them.

The diagnosis therefore does not use such columns. The route `descriptors` takes only what the
program computes from the structure. The columns of the file are scored beside it, as the route
`file columns`, which is printed and never recommended. If a column of a file is a real
descriptor, for example the temperature of the measurement, it is named, and then it joins the
baseline: `diagnose(..., descriptors=["T"])`, or `--descriptors T` on the command line.

The recommendation is `none`. That is a result. It says that this table, as it is, does not
support a model, and it says so in three seconds, before a week is spent on fields and frames.

### Why the columns are left out and not tested

An earlier form of the diagnosis took all columns of the file into the baseline and then looked
for a leak: it refused the baseline when it reached a q² of 0.95. That is a threshold, and a
threshold can be passed. The cell shows the case that passed it, a related table of corrosion
inhibitors with a pyrazole ring, which has the same columns. The diagnosis is run on all 25
molecules and on 17 of them.

In [7]:
pyrazoles = load_dataset("examples/corrosion/13_c1c_nH_nn1.csv")
y_pyrazoles = np.asarray(pyrazoles.activity, float)
part = sorted(np.random.RandomState(3).permutation(len(y_pyrazoles))[8:].tolist())      # 17 of the 25

print("%-18s %22s %14s   %s" % ("", "computed descriptors", "file columns", "recommended"))
for label, rows in (("all 25 molecules", list(range(len(y_pyrazoles)))), ("17 of them", part)):
    d = diagnose([pyrazoles.mols[i] for i in rows], [pyrazoles.names[i] for i in rows], y_pyrazoles[rows],
                 routes=ROUTES, activity_prop=pyrazoles.activity_prop)
    print("%-18s %+22.2f %+14.2f   %s" % (label, d.routes["descriptors"].q2, d.routes["file columns"].q2, d.recommended))

                     computed descriptors   file columns   recommended


all 25 molecules                    +0.35          +0.98   descriptors


17 of them                          +0.18          +0.94   invariant


On the 17 molecules the columns of the file give a q² of 0.94, a little below the old threshold.
The old diagnosis gave no warning there and recommended descriptors, which meant: predict the
activity from the activity. Now those columns do not enter the baseline, whatever they score,
so no threshold decides anything.

The two recommendations of this cell differ from each other, and that is a second lesson. With
17 to 25 molecules a recommendation is not stable; the report says how wide the intervals are.

## Searching harder is not a better model

Every route in the reports above chose its settings inside each training part. The option
`nested=False` switches that off: each route is then scored at the best point of its own grid,
which is how a quick comparison is often made. The cell runs the ACE set both ways.

In [8]:
flat = diagnose(ace.mols, ace.names, y_ace, charge_model="file", routes=ROUTES,
                activity_prop=ace.activity_prop, nested=False)
print("%-12s %28s %30s %12s" % ("route", "settings chosen inside", "best point of its own grid", "difference"))
for name in ROUTES:
    inside, best = reports["ACE"].routes[name].q2, flat.routes[name].q2
    print("%-12s %+28.3f %+30.3f %+12.3f" % (name, inside, best, best - inside))

route              settings chosen inside     best point of its own grid   difference
aligned                            +0.605                         +0.657       +0.052
counts                             +0.660                         +0.654       -0.007
invariant                          +0.677                         +0.680       +0.003
descriptors                        +0.644                         +0.633       -0.011


The aligned route gains most from being scored at its best point, because it searches the most
settings: six attenuations and five ridge parameters. The other routes search the ridge
parameter only, and they change by about 0.01. So a comparison at the best points favours the route with
the largest grid, and with it the aligned model would look better against its references than
it is. This is the same fault as a reference that is not searched, seen from the other side.

## What the advice is worth

A tool that reports four baselines is useful whatever its last line says. Whether the last
line, the recommendation, is good advice has to be measured. The repository has done that
(`python benchmarks/run_diagnose.py --validate`): 30 % of a set is held out, the diagnosis is
run on the other 70 %, the model that it recommends is built, and that model is scored on the
molecules that were held out, against every other route. Thirteen sets, three splits each. The
cell reads the stored results.

In [9]:
import glob
import sys

sys.path.insert(0, "benchmarks")
import run_diagnose                               # the benchmark script; it holds the summary

stored = []
for path in sorted(glob.glob("benchmarks/results/diagnose/*_validate.json")):
    with open(path, encoding="utf-8") as fh:
        stored.append(json.load(fh))
splits = run_diagnose.validation_splits(stored)
s = run_diagnose.advice_summary(splits)
print("%d splits of %d sets" % (s["splits"], s["sets"]))
print("the advised route was the best of the five in %d splits; it fell %.3f in R2 short of the best on the average" % (s["advice_best"], s["advice_cost"]))
print("the aligned route was the best in %d splits; always taking it costs %.3f" % (s["aligned_best"], s["aligned_cost"]))
print("the advice minus the aligned route, mean over the sets: %+.3f [%+.3f, %+.3f]" % (s["gain"], s["gain_low"], s["gain_high"]))
print("sets on which the advice is better: %d, worse: %d" % (s["sets_better"], s["sets_worse"]))
print("splits in which a route that reads the activity was advised:", sum(x["leak"] for x in splits))
print("splits in which the advice was that no route is worth a model:", sum(x["no_route"] for x in splits))

39 splits of 13 sets
the advised route was the best of the five in 16 splits; it fell 0.117 in R2 short of the best on the average
the aligned route was the best in 12 splits; always taking it costs 0.175
the advice minus the aligned route, mean over the sets: +0.058 [-0.048, +0.225]
sets on which the advice is better: 5, worse: 5
splits in which a route that reads the activity was advised: 0
splits in which the advice was that no route is worth a model: 3


The advice names the best route in fewer than half of the splits. Following it is better than
always taking the aligned model by 0.06 in R² on the average over the sets, and the interval of
that difference contains zero. It is better on five sets and worse on five.

So the recommendation is **insurance and not an advantage**. It costs a little on sets that are
well aligned, where the aligned model was right all along. It protects against the case in
which an automatic frame means nothing: on one corrosion table the aligned model falls to an R²
of -1.26 on held-out molecules, and the advised route reaches 0.57. It is not a way to a better
model on the average.

The line before the last belongs to the section on the columns of a file. With the earlier
form of the diagnosis five of these splits advised descriptors that were the activity in
another form, and their scores near 1 were counted as the largest successes of the advice. The
gain then looked twice as large as it is. The summary still looks for such a split, and it
finds none.

The last line counts the thiadiazoles. On each of their three splits the advice was that no
route is worth a model. The summary scores such a split as the aligned model, because that is
what a user who builds a model anyway would build. In two of the three no route reached an R² of
0.05 on the molecules that were held out.

## What to take from this notebook

- Run the diagnosis before the model. It takes seconds to a few minutes.
- Read the lines and not only the last one. A warning is a question that the set has not
  answered.
- `aligned` is recommended only when the frame carries signal and the molecules are really
  superimposed. `invariant` often means that the frame is not worth the work, not that the
  invariant kernel predicts better. `descriptors` and `counts` mean that the geometry adds
  nothing to the prediction. `none` means that the table does not support a model.
- The columns of a file are not descriptors until you name them. They can be the activity in
  another form, and no test of a score can tell.
- Compare routes with their settings chosen inside the folds, and give a reference route the
  settings it needs. The diagnosis does the first, and for the invariant route the second is
  left to you.
- With twenty molecules a recommendation can change when a few of them are left out.

From the command line:

```
cmf diagnose examples/sutherland/THR.sdf --charges file
cmf diagnose examples/corrosion/7_c1nncs1.csv --json diagnosis.json
```

Where to read further in the manual: the chapter *Before you model: `cmf diagnose`* has the five
questions with the figure of the decisions, the two protections against a leak and against a
search that is too hard, and the measurement of the advice. *Three tests to run before you build
a new field or a new score* asks the same kind of question of a new idea and not of a data set.